# 07.03 · Control flow: `Command`, `Send`, cycles, recursion limits

**Module 07 · LangGraph Fundamentals** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 120–150 min |
| 📋 Prerequisites | Lesson 07.02. |
| 🔑 Keys | none |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Update state and route in one step** with `Command(update=..., goto=...)`, and declare `destinations` so the graph stays drawable.
2. **Fan out dynamically** (map-reduce) with `Send`: one worker per item, results merged by a reducer.
3. **Build a cycle** that always ends: a counter in state, a conditional exit.
4. **Bound everything** with `recursion_limit`, catch `GraphRecursionError`, and degrade gracefully with `RemainingSteps`.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Use the Graph API](https://docs.langchain.com/oss/python/langgraph/use-graph-api) | `Command`, map-reduce with `Send`, loops, recursion limit, `defer=True` |
| [Graph API overview](https://docs.langchain.com/oss/python/langgraph/graph-api) | Super-steps, `Send`, `Command`, recursion limit and `RemainingSteps` |
| [LangGraph runtime (Pregel)](https://docs.langchain.com/oss/python/langgraph/pregel) | What runs under the hood: channels, super-steps, why parallel writes need reducers |
| [GRAPH_RECURSION_LIMIT](https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT) | The error, and how to fix the loop rather than raise the limit |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langgraph",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/07-langgraph-fundamentals/).
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (07.07 reuses Module 06): no usage telemetry.
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

---
## Part 1 · `Command`: update and go, from inside the node

A conditional edge routes **after** a node, using a separate function. Sometimes the node itself knows where to go next, and wants to write state at the same time. Return `Command(update={...}, goto="node")`. Tell `add_node` where it may go (`destinations=`) so the graph can still be drawn.

In [ ]:
# 🧪 Offline: a refund desk. The node decides the next step and records why, in one return value.
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Command


class Refund(TypedDict, total=False):
    amount: float
    days_since_delivery: int
    decision: str
    reason: str


def assess(state: Refund) -> Command[Literal["auto_refund", "human_review", "reject"]]:
    if state["days_since_delivery"] > 30:
        return Command(update={"reason": "outside the 30-day window"}, goto="reject")
    if state["amount"] > 300:
        return Command(update={"reason": "large amount"}, goto="human_review")
    return Command(update={"reason": "small, in window"}, goto="auto_refund")


def auto_refund(state: Refund) -> dict:
    return {"decision": "refunded"}


def human_review(state: Refund) -> dict:
    return {"decision": "queued for a person"}


def reject(state: Refund) -> dict:
    return {"decision": "rejected"}


rb = StateGraph(Refund)
rb.add_node(assess)                       # the Command[Literal[...]] return type tells LangGraph the destinations
for node in (auto_refund, human_review, reject):
    rb.add_node(node)
    rb.add_edge(node.__name__, END)
rb.add_edge(START, "assess")
refunds = rb.compile()

for case in ({"amount": 49, "days_since_delivery": 3}, {"amount": 349, "days_since_delivery": 10},
             {"amount": 49, "days_since_delivery": 45}):
    out = refunds.invoke(case)
    print(f"{case} → {out['decision']} ({out['reason']})")
print("drawn edges from assess:", refunds.get_graph().draw_mermaid().count("assess -.->"))

**You should now see** `refunded (small, in window)`, `queued for a person (large amount)`, `rejected (outside the 30-day window)`, and `drawn edges from assess: 3`.

| Use | When |
|---|---|
| Conditional edge (`add_conditional_edges`) | Routing is a pure decision on state; you want routing logic separate and testable |
| `Command(update, goto)` | The node computes something **and** decides where to go; or a subgraph/agent hands off to another node (`graph=Command.PARENT`, Module 09) |

Without the `Command[Literal[...]]` annotation (or `add_node(..., destinations=(...))`), the graph still runs, but the drawing can't show `assess`'s edges.

---
## Part 2 · `Send`: map-reduce over a list you only know at run time

Edges are fixed when you compile. But "research **each** section of an outline" has as many branches as the outline has sections. A conditional edge can return a **list of `Send("node", payload)`**: LangGraph runs `node` once per payload, **in parallel** (same super-step), each with its own private input. Their updates meet in a reducer.

In [ ]:
# 🧪 Offline: fan out one worker per product, reduce with operator.add, then summarize.
import operator
import time
from typing import Annotated

from langgraph.types import Send

PRICES = {"Nimbus Mini": 89, "Nimbus S1": 149, "Nimbus S2": 199, "Nimbus Pro": 349}


class Survey(TypedDict, total=False):
    products: list[str]
    findings: Annotated[list[dict], operator.add]     # every worker appends here
    summary: str


def fan_out(state: Survey) -> list[Send]:
    return [Send("check_price", {"product": p}) for p in state["products"]]


def check_price(task: dict) -> dict:          # receives the Send payload, NOT the graph state
    time.sleep(0.2)                           # pretend this is a slow API call
    return {"findings": [{"product": task["product"], "price": PRICES[task["product"]]}]}


def summarize(state: Survey) -> dict:
    cheapest = min(state["findings"], key=lambda f: f["price"])
    return {"summary": f"{len(state['findings'])} prices checked; cheapest: {cheapest['product']} (US${cheapest['price']})"}


sb = StateGraph(Survey)
sb.add_node(check_price)
sb.add_node(summarize)
sb.add_conditional_edges(START, fan_out, ["check_price"])
sb.add_edge("check_price", "summarize")
sb.add_edge("summarize", END)
survey = sb.compile()

start = time.perf_counter()
out = survey.invoke({"products": list(PRICES)})
print(out["summary"], f"| {time.perf_counter() - start:.1f}s for 4 × 0.2s workers")
print(sorted(f["product"] for f in out["findings"]))

**You should now see** `4 prices checked; cheapest: Nimbus Mini (US$89) | 0.2s for 4 × 0.2s workers` (about 0.2–0.3 s, not 0.8 s: the workers ran in parallel threads) and the four product names.

### What just happened?

`fan_out` returned four `Send`s. All four `check_price` tasks ran in **one super-step**, each seeing only its payload. Their four `findings` updates were combined by `operator.add` (without a reducer you'd get the `InvalidUpdateError` from 07.02). `summarize` ran once, in the next super-step, after **all** workers finished. That's the orchestrator-worker pattern of 07.04 and the research step of Project 07.

### In which order do parallel results land?

Workers finish in whatever order the outside world allows. Predict before you run the next cell: five workers, where worker 0 is the **slowest** and worker 4 the **fastest**. In which order do their updates **stream** out, and in which order are they in the **final state**?

In [ ]:
# 🧪 Offline: streamed order vs final-state order for parallel workers.
class Race(TypedDict):
    out: Annotated[list[int], operator.add]


def runner(task: dict) -> dict:
    time.sleep(0.05 * (4 - task["i"]))        # worker 0 sleeps longest, worker 4 not at all
    return {"out": [task["i"]]}


rb = StateGraph(Race)
rb.add_node(runner)
rb.add_conditional_edges(START, lambda s: [Send("runner", {"i": i}) for i in range(5)], ["runner"])
rb.add_edge("runner", END)
race = rb.compile()

streamed = [u["runner"]["out"][0] for u in race.stream({"out": []}, stream_mode="updates")]
print("streamed updates (as workers finish):", streamed)
print("final state                         :", race.invoke({"out": []})["out"])

**You should now see** the streamed updates in finishing order, `[4, 3, 2, 1, 0]`, but the final state in `Send` order, `[0, 1, 2, 3, 4]`.

**What just happened?** Streaming shows each worker's update **as it finishes**. The final state is different: at the end of the super-step, this version of LangGraph applies the workers' writes in a fixed task order (the order of the `Send`s), not in completion order. Don't build on that detail: the docs make no promise about it ("updates from a parallel superstep may not be ordered consistently") and recommend storing a value to order by. So when order matters, **carry an index in the payload and sort on it**. Lesson 07.04's orchestrator and Project 07 do exactly that.

---
## Part 3 · Cycles that always end

A graph may loop: an edge back to an earlier node. LangGraph doesn't stop a loop for you, so **you** put the stop condition in state. Here, a "polisher" shortens a slogan until it fits, at most 3 tries.

In [ ]:
# 🧪 Offline: a bounded cycle. Two exits: success, or the attempt counter.
class Slogan(TypedDict, total=False):
    text: str
    attempts: int


MAX_ATTEMPTS = 3


def shorten(state: Slogan) -> dict:
    words = state["text"].split()
    return {"text": " ".join(words[:-1]), "attempts": state.get("attempts", 0) + 1}


def fits_or_give_up(state: Slogan) -> str:
    if len(state["text"]) <= 30:
        return "done"
    if state["attempts"] >= MAX_ATTEMPTS:
        return "give_up"
    return "again"


lb = StateGraph(Slogan)
lb.add_node(shorten)
lb.add_edge(START, "shorten")
lb.add_conditional_edges("shorten", fits_or_give_up, {"again": "shorten", "done": END, "give_up": END})
slogans = lb.compile()

print(slogans.invoke({"text": "Weather you can trust, every single morning"}))
print(slogans.invoke({"text": "The most accurate and dependable home weather station you will ever own, guaranteed"}))

**You should now see** the first slogan shortened to `Weather you can trust, every` (28 characters, 2 attempts), and the second stopped at `attempts: 3`, still too long. Both runs **ended**: the counter guarantees it, whatever the "model" does.

`mastery.graphs.loop_router(score_key=..., threshold=..., max_iterations=..., again=..., done=...)` is this pattern packaged for evaluator-optimizer loops (07.04, Project 07).

---
## Part 4 · The safety net: `recursion_limit`

Every run has a **recursion limit**: the maximum number of super-steps. When it's reached, LangGraph raises `GraphRecursionError`. It's a net for bugs (a loop whose exit never fires), not a design tool.

In [ ]:
# 🧪 Offline: a loop with a bug (it never shortens), stopped by recursion_limit.
from langgraph.errors import GraphRecursionError
from langgraph._internal._config import DEFAULT_RECURSION_LIMIT   # internal: printed for information only


def broken_shorten(state: Slogan) -> dict:
    return {"attempts": state.get("attempts", 0) + 1}          # bug: text never changes


bb = StateGraph(Slogan)
bb.add_node("shorten", broken_shorten)
bb.add_edge(START, "shorten")
bb.add_conditional_edges("shorten", lambda s: "again" if len(s["text"]) > 30 else "done", {"again": "shorten", "done": END})
broken = bb.compile()

try:
    broken.invoke({"text": "x" * 50}, {"recursion_limit": 10})
except GraphRecursionError as exc:
    print("GraphRecursionError:", str(exc).splitlines()[0][:90])
print("default limit in this langgraph:", DEFAULT_RECURSION_LIMIT)

**You should now see** `GraphRecursionError: Recursion limit of 10 reached without hitting a stop condition...` and `default limit in this langgraph: 10007`.

> ⚠️ **Docs vs code (checked 2026-10-12):** the Graph API page says the default limit is **1000** since 1.0.6; langgraph **1.2.14** ships **10007** (`LANGGRAPH_DEFAULT_RECURSION_LIMIT` env var overrides it). Either way it's far too high to protect your token budget: a buggy loop with a model call per step could make thousands of calls. **Always pass your own `recursion_limit`** for graphs with cycles. `recursion_limit` is a top-level config key, **not** inside `configurable`.

**Graceful degradation.** Instead of crashing at the limit, a node can read how many steps are left: add `remaining_steps: RemainingSteps` to the state (LangGraph fills it in) and stop on your own terms.

In [ ]:
# 🧪 Offline: stop cleanly two steps before the limit, with a partial result.
from langgraph.managed import RemainingSteps


class Budgeted(TypedDict, total=False):
    attempts: int
    status: str
    remaining_steps: RemainingSteps


def work(state: Budgeted) -> Command[Literal["work", "__end__"]]:
    if state["remaining_steps"] <= 2:
        return Command(update={"status": f"stopped early after {state.get('attempts', 0)} attempts"}, goto=END)
    return Command(update={"attempts": state.get("attempts", 0) + 1}, goto="work")


gb = StateGraph(Budgeted)
gb.add_node(work)
gb.add_edge(START, "work")
print(gb.compile().invoke({}, {"recursion_limit": 8}))

**You should now see** `{'attempts': 5, 'status': 'stopped early after 5 attempts'}`: no exception, and the caller gets a usable result. (`remaining_steps` itself isn't in the output: it's a managed value, not stored state.)

## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `GraphRecursionError` | A loop's exit condition never fires | Fix the condition / add a counter; don't just raise the limit |
| `recursion_limit` seems ignored | Put inside `configurable` | `graph.invoke(x, {"recursion_limit": 10})` (top level) |
| `InvalidUpdateError` after adding `Send` | Workers write a key with no reducer | `Annotated[list, operator.add]` on the collected key |
| `KeyError` in a `Send` worker | The worker gets the **payload**, not the state | Put everything it needs in the `Send` payload |
| Router edges missing from the drawing | `Command` without destinations | `-> Command[Literal["a", "b"]]` or `add_node(..., destinations=("a", "b"))` |
| Fan-in node runs too early / twice | Branches of different lengths join | `add_edge(["a", "b"], "join")` waits for both; `defer=True` waits for everything pending (docs) |

## ⚠️ Production notes

> ⚠️ **Production note: bound fan-out, too.**
> `Send` makes one task per item: a 500-item list means 500 parallel model calls and a rate-limit storm. Cap the list, batch items, or limit concurrency (`max_concurrency` in the run config).

> ⚠️ **Production note: three limits, three jobs.**
> A counter in state ends loops by design; `RemainingSteps` degrades gracefully; `recursion_limit` catches bugs. Set all three, and log which one stopped a run.

## 🏋️ Exercises

1. **(Easy)** Add a fourth refund rule to `assess`: amounts under US$10 are refunded without checking the window.
2. **(Medium)** Change `fan_out` to skip products over US$300, and make `summarize` handle an empty `findings` list.
3. **(Medium)** In `slogans`, what is the smallest `recursion_limit` that still lets the second slogan finish its 3 attempts? Try it.
4. **(Stretch)** Rewrite the slogan loop with `Command` inside `shorten` (no conditional edge). Which version is easier to test?

<details>
<summary><b>Show solutions</b></summary>

**1.** First line of `assess`: `if state["amount"] < 10: return Command(update={"reason": "tiny amount"}, goto="auto_refund")`.

**2.** `[Send(...) for p in state["products"] if PRICES[p] <= 300]`; in `summarize`: `if not state.get("findings"): return {"summary": "nothing to compare"}`. (If the list is empty, no worker runs and the edge to `summarize` never fires: route `START` to `summarize` in that case, e.g. return `"summarize"` from `fan_out` when there's nothing to send.)

**3.** Measured: `recursion_limit=4` works, `3` raises `GraphRecursionError`. So three `shorten` steps need a limit of 4: the limit is checked against LangGraph's step counter (`config["metadata"]["langgraph_step"]`), which doesn't map one-to-one to your node count. Leave headroom and test the limit you choose.

**4.** `return Command(update=..., goto=END if fits else "shorten")` with `-> Command[Literal["shorten", "__end__"]]`. The conditional-edge version is easier to unit-test (the router is a pure function); the `Command` version keeps the logic in one place.
</details>

## ✅ Checkpoint

- [ ] I can route with `Command` and keep the graph drawable.
- [ ] I built a map-reduce with `Send` and a reducer.
- [ ] Every cycle I write has a counter **and** I set `recursion_limit`.
- [ ] I can catch `GraphRecursionError` and use `RemainingSteps`.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

The default recursion limit differs between the docs (1000) and langgraph 1.2.14 (10007). Check the [changelog](https://docs.langchain.com/oss/python/releases/changelog) or the [Graph API page](https://docs.langchain.com/oss/python/langgraph/graph-api#recursion-limit) for your installed version (`python -c "from langgraph._internal._config import DEFAULT_RECURSION_LIMIT as d; print(d)"`) and log what you find in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- `Command(update, goto)` = write state and choose the next node in one return; declare destinations.
- `Send` = dynamic fan-out; workers get payloads, results meet in a reducer, the next node waits for all.
- Loops end because **you** count; `recursion_limit` (top-level config) is the net; `RemainingSteps` lets you stop gracefully.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 07.04 · Workflow patterns** → [`07-04-workflow-patterns.ipynb`](07-04-workflow-patterns.ipynb).

[Module 07 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)